# 27 Local Baseline ML Modeling
Local-only baseline modeling from Phase 26 features.


## 1) Load Data and Inspect Actual Schema


In [ ]:
from pathlib import Path
import json
from datetime import datetime, timezone
import pandas as pd

from src.modeling.baseline_model import (
    ModelingConfig,
    build_column_groups,
    run_baseline_modeling,
    write_modeling_artifacts,
)

feature_path = Path('local/derived/features/bluesky_engagement_features.parquet')
feature_summary_path = Path('local/derived/features/bluesky_engagement_feature_summary.json')

feature_df = pd.read_parquet(feature_path)
feature_summary = None
if feature_summary_path.exists():
    payload = json.loads(feature_summary_path.read_text())
    feature_summary = payload.get('summary')

print('rows:', len(feature_df), 'cols:', len(feature_df.columns))
print('target distribution:')
print(feature_df['engagement_label'].astype(str).value_counts())


In [ ]:
print('columns:')
print(feature_df.columns.tolist())


## 2) Define Target / Model / Debug Columns and Exclusions


In [ ]:
groups = build_column_groups(feature_df, feature_summary=feature_summary)
print('target:', groups['target_column'])
print('model feature count:', len(groups['model_feature_columns']))
print('debug column count:', len(groups['debug_columns']))
print('label columns:', groups['label_columns'])
print('excluded columns:', groups['excluded_columns'])


## 3) Train and Evaluate Baselines


In [ ]:
config = ModelingConfig(
    random_seed=42,
    test_size=0.30,
    logistic_learning_rate=0.10,
    logistic_max_iter=1200,
    logistic_l2=0.001,
    stump_estimators=200,
    stump_max_features=None,
    stump_threshold_count=9,
)

modeling_output = run_baseline_modeling(
    feature_df=feature_df,
    feature_summary=feature_summary,
    config=config,
)

results = modeling_output['results']
print('split:', results['split'])
print('best_model:', results['best_model_name'])
print('criterion:', results['best_model_criterion'])


In [ ]:
softmax_metrics = results['models']['softmax_regression']['metrics']
stump_metrics = results['models']['bagged_stump_ensemble']['metrics']
print('softmax metrics:', softmax_metrics)
print()
print('stump metrics:', stump_metrics)


## 4) Confusion Matrix and Error Analysis


In [ ]:
confusion_df = results['confusion_matrix']
confusion_df


In [ ]:
pred_df = results['prediction_sample']
print('prediction sample rows:', len(pred_df))
print('incorrect predictions:', int((~pred_df['is_correct']).sum()))
pred_df[['uri','true_label','predicted_label','best_model_name','is_correct']].head(20)


In [ ]:
if 'true_label' in pred_df.columns and 'predicted_label' in pred_df.columns:
    confusion_pairs = (
        pred_df.loc[pred_df['true_label'] != pred_df['predicted_label'], ['true_label', 'predicted_label']]
        .value_counts()
    )
    print('common confusion pairs:')
    print(confusion_pairs if len(confusion_pairs) else 'none')


## 5) Feature Importance / Interpretability


In [ ]:
importance_df = results['feature_importance']
print('importance rows:', len(importance_df))
importance_df.head(20)


In [ ]:
print('top softmax (__all__) features:')
print(
    importance_df.loc[
        (importance_df['model_name'] == 'softmax_regression')
        & (importance_df['class_label'] == '__all__')
    ]
    .head(15)
    [['feature_name','importance','signed_value']]
    .to_string(index=False)
)
print()
print('top stump features:')
print(
    importance_df.loc[importance_df['model_name'] == 'bagged_stump_ensemble']
    .head(15)
    [['feature_name','importance']]
    .to_string(index=False)
)


## 6) Write Phase 27 Artifacts


In [ ]:
artifact_paths = write_modeling_artifacts(
    modeling_output=modeling_output,
    output_dir='local/derived/modeling',
    sample_csv_path='data/samples/baseline_predictions_sample_1000.csv',
)
artifact_paths


## 7) Readiness Statement
Phase 27 is complete when metrics, confusion matrix, feature importance, and prediction sample artifacts are written and documented.
